In [19]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris

# Load Iris dataset from sklearn
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [26]:
import numpy as np

print("--- Iris Dataset on KNN using NumPy (Manhattan) ---")

def ManhattanDistance(x1, x2):
    return np.sum(np.abs(x1 - x2))

class KNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X):
        predictions = [self._predict(x) for x in X]
        return np.array(predictions)

    def _predict(self, x):
        distances = [ManhattanDistance(x_row, x) for x_row in self.X]
        indices = np.argsort(distances)[:self.k]
        return self.Most_Common_Label(self.y[indices])

    def Most_Common_Label(self, y):
        labels, counts = np.unique(y, return_counts=True)
        return labels[np.argmax(counts)]

def accuracy(y_test, y_pred):
    return np.sum(y_test == y_pred) / len(y_test)

def f1_score_numpy(y_test, y_pred):
    labels = np.unique(np.concatenate((y_test, y_pred)))
    f1_scores = []
    for label in labels:
        tp = np.sum((y_test == label) & (y_pred == label))
        fp = np.sum((y_test != label) & (y_pred == label))
        fn = np.sum((y_test == label) & (y_pred != label))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)
    return np.mean(f1_scores)

# k=3, Manhattan Distance
model = KNN(k=3)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print("Predictions (first 10):", y_pred[:10])
print("\nActual Test values (first 10):", y_test[:10])
print(f"\nAccuracy is: {accuracy(y_test, y_pred):.2%}")
print(f"F1 Score is: {f1_score_numpy(y_test, y_pred):.2%}")

--- Iris Dataset on KNN using NumPy (Manhattan) ---
Predictions (first 10): [0 1 1 1 2 0 0 0 1 2]

Actual Test values (first 10): [0 1 1 1 2 0 0 0 1 2]

Accuracy is: 90.00%
F1 Score is: 88.24%


In [27]:
print("--- Iris Dataset on KNN using Scikit-Learn (Manhattan) ---")
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

# n_neighbors=3, metric='manhattan'
clf = KNeighborsClassifier(n_neighbors=3, metric='manhattan')
clf.fit(X_train_scaled, y_train)
predictions = clf.predict(X_test_scaled)

print("Predictions (first 10):", predictions[:10])
print("\nActual test values (first 10):", y_test[:10])
print(f"\nAccuracy: {accuracy_score(y_test, predictions):.2%}")
print(f"F1 Score (macro): {f1_score(y_test, predictions, average='macro'):.2%}")

--- Iris Dataset on KNN using Scikit-Learn (Manhattan) ---
Predictions (first 10): [0 1 1 1 2 0 0 0 1 2]

Actual test values (first 10): [0 1 1 1 2 0 0 0 1 2]

Accuracy: 90.00%
F1 Score (macro): 88.24%


In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris

# 1. Load Data
iris = load_iris()
X = iris.data
y = iris.target

# 2. Split (No random_state)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 3. Model
# K=3, Manhattan Distance (p=1), Weighted by distance
clf = KNeighborsClassifier(n_neighbors=3,
                           weights='distance',
                           algorithm='ball_tree',
                           leaf_size=10,
                           p=1)

clf.fit(X_train, y_train)
predictions = clf.predict(X_test)

print("Predictions (first 10):", predictions[:10])
print(f"Scikit-Learn Accuracy: {accuracy_score(y_test, predictions) * 100}%")

Predictions (first 10): [0 0 2 0 0 0 1 1 0 2]
Scikit-Learn Accuracy: 90.0%


In [29]:
import numpy as np
from collections import Counter
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

class OptimizedKNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X_test):
        predictions = [self._predict_single(x) for x in X_test]
        return np.array(predictions)

    def _predict_single(self, x):
        # 1. Calculate MANHATTAN Distance (p=1)
        distances = np.sum(np.abs(self.X_train - x), axis=1)

        # 2. Get indices of the k nearest neighbors
        k_indices = np.argsort(distances)[:self.k]
        k_nearest_labels = self.y_train[k_indices]
        k_nearest_distances = distances[k_indices]

        # 3. Perform WEIGHTED Voting (Inverse of distance)
        weights = []
        for d in k_nearest_distances:
            if d == 0:
                weights.append(1e9)
            else:
                weights.append(1 / d)

        # Tally the weighted votes
        vote_counts = {}
        for weight, label in zip(weights, k_nearest_labels):
            vote_counts[label] = vote_counts.get(label, 0) + weight

        return max(vote_counts, key=vote_counts.get)

# --- Testing the Manual Code ---

# Load Data
iris = load_iris()
X = iris.data
y = iris.target

# Split (No random_state)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Run Model with K=3
my_knn = OptimizedKNN(k=3)
my_knn.fit(X_train, y_train)
manual_preds = my_knn.predict(X_test)
print("Predictions (first 10):", manual_preds[:10])

accuracy = np.mean(manual_preds == y_test)
print(f"NumPy (Manual) Optimized Accuracy: {accuracy * 100}%")

Predictions (first 10): [0 0 0 1 1 1 1 1 2 0]
NumPy (Manual) Optimized Accuracy: 90.0%
